# 01 — Raw data analysis (Step 1)
**Team 8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**

Goal of this notebook (paper step 1 — *raw data visualization/analysis*): load the **ACC Arena** raw
data, build a tidy per-user/per-timestamp table, and inspect it to decide which features look promising
for predicting **throughput**.

> The raw dataset is in **wide** format: one folder per metric (Throughput, BLER, PRB, RU, SINR DL/UL,
> Positions), each CSV holding `time` + one column per user (`entityStats id N`), ~500 users per file.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # every output row aggregates all raw samples in a 120-s window
N_USERS          = None          # modeling notebooks use all ~12k ACC Arena users
EDA_USERS        = 500           # seeded population-wide sample used only for fast exploratory plots
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=0/1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
OUTLIER_PCT      = None          # optional train-only sensitivity analysis; primary results keep the full distribution.
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = False          # primary task covers every user; True is an optional active-only sensitivity run
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw wide format -> uniform tidy windows) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(path):
            first, last = file_id_range(path)
            return any(first <= user <= last for user in user_ids)
        files = [path for path in files if overlaps(path)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    ids = []
    for path in metric_files(venue_dir, "*Throughput*", "*.csv"):
        first, last = file_id_range(path)
        ids.extend(range(first, last + 1))
    return np.asarray(sorted(ids))

def _time_origin(venue_dir):
    """Common origin shared by both raw timelines in a venue."""
    starts = []
    for path in glob.glob(os.path.join(venue_dir, "**", "*.csv"), recursive=True):
        starts.append(float(pd.read_csv(path, usecols=[0], nrows=1).iloc[0, 0]))
    return min(starts)

def _window_index(times, origin, seconds):
    return origin + np.floor((np.asarray(times, dtype=float) - origin) / seconds) * seconds

def load_metric(files, value_name, origin, seconds, user_ids, how="mean"):
    """Aggregate every raw observation into a deterministic, uniform time window."""
    out = []
    for path in files:
        header = list(pd.read_csv(path, nrows=0).columns)
        column_to_user = {c: int(re.search(r'(\d+)', c).group(1)) for c in header[1:]}
        usecols = [header[0]] + [c for c in header[1:] if column_to_user[c] in user_ids]
        wide = pd.read_csv(path, usecols=usecols).rename(columns={header[0]: "raw_time"})
        wide["time"] = _window_index(wide.pop("raw_time"), origin, seconds)
        values = wide.groupby("time", sort=True)
        wide = (values.mean() if how == "mean" else values.last()).astype("float32")
        wide = wide.rename(columns=column_to_user)
        out.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, origin, seconds, user_ids):
    """Aggregate positions per window and convert latitude/longitude to local metres."""
    frames = []
    for path in files:
        first = pd.read_csv(path, nrows=1).values.astype(float)
        all_ids = first[0, 1::5].astype(int)
        blocks = [k for k, user in enumerate(all_ids) if user in user_ids]
        if not blocks:
            continue
        usecols = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]
        raw = pd.read_csv(path, usecols=usecols).values.astype(float)
        ids = raw[0, 1::5].astype(int)
        bins = _window_index(raw[:, 0], origin, seconds)
        pieces = []
        for name, values, how in [
            ("lat", raw[:, 2::5], "mean"), ("lon", raw[:, 3::5], "mean"),
            ("z", raw[:, 4::5], "mean"), ("traffic_type", raw[:, 5::5], "last")]:
            wide = pd.DataFrame(values, index=bins, columns=ids)
            wide = wide.groupby(level=0, sort=True)
            wide = wide.mean() if how == "mean" else wide.last()
            wide.index.name = "time"
            pieces.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=name))
        frame = pieces[0]
        for piece in pieces[1:]:
            frame = frame.merge(piece, on=["time", "user_id"], validate="one_to_one")
        frames.append(frame)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()
    radius_m = 6_371_000.0
    pos["x"] = radius_m * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = radius_m * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    venue = find_venue_dir(DATA_ROOT, venue_key)
    population = all_user_ids(venue)
    if n_users is None:
        user_ids = set(map(int, population))
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(map(int, rng.choice(population, size=min(n_users, len(population)), replace=False)))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(population)}")
    else:
        user_ids = set(map(int, population[:n_users]))
    origin = _time_origin(venue)
    mf = lambda sub, pattern: metric_files(venue, sub, pattern, user_ids)
    parts = [
        load_metric(mf("*Throughput*", "*.csv"), "throughput", origin, resample_seconds, user_ids),
        load_metric(mf("*BLER*", "*.csv"), "bler", origin, resample_seconds, user_ids),
        load_metric(mf("*PRB*", "*.csv"), "prb", origin, resample_seconds, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"), "ru_id", origin, resample_seconds, user_ids, how="last"),
        load_metric(mf("*SINR*", "SINRDL_*.csv"), "sinr_dl", origin, resample_seconds, user_ids),
        load_metric(mf("*SINR*", "SINRUL_*.csv"), "sinr_ul", origin, resample_seconds, user_ids),
        load_positions(mf("*Positions*", "*.csv"), origin, resample_seconds, user_ids),
    ]
    frame = parts[0]
    for part in parts[1:]:
        frame = frame.merge(part, on=["time", "user_id"], how="inner", validate="one_to_one")
    frame = frame.dropna().reset_index(drop=True)
    frame["user_id"] = frame.user_id.astype(int)
    frame["traffic_type"] = frame.traffic_type.round().astype(int)
    frame["ru_id"] = frame.ru_id.round().astype(int)
    assert not frame.duplicated(["time", "user_id"]).any()
    return frame


## Load a representative EDA sample
The modeling pipeline uses all 12,000 ACC users. For interactive EDA only, we draw a seeded sample across the full user-ID population; every temporal observation for those users contributes to the 120-second windows.

In [ ]:
df = assemble("acc", n_users=EDA_USERS, resample_seconds=RESAMPLE_SECONDS, random_users=True)
print("EDA shape:", df.shape, "| users:", df.user_id.nunique(), "| uniform windows:", df.time.nunique())
df.head()

In [ ]:
df.describe()

## Sampling quality and uniform temporal windows
The raw cadence is not perfectly regular. The left panel makes the jitter explicit; the right panel shows the aggregated, uniformly spaced analysis grid.

In [ ]:
# === Sampling jitter and the uniform-window solution ===
from collections import Counter
import matplotlib.pyplot as plt

venue = find_venue_dir(DATA_ROOT, "acc")
reference_files = {
    "Throughput / PRB": metric_files(venue, "*Throughput*", "*.csv")[:1],
    "BLER / SINR / RU / position": metric_files(venue, "*BLER*", "*.csv")[:1],
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, paths in reference_files.items():
    raw_time = pd.read_csv(paths[0], usecols=[0]).iloc[:, 0].to_numpy(float)
    delta = np.diff(raw_time)
    values, counts = np.unique(delta, return_counts=True)
    axes[0].bar(values + (0.08 if label.startswith("Throughput") else -0.08), counts,
                width=0.16, alpha=0.8, label=label)
axes[0].set_xlabel("Consecutive timestamp difference (s)")
axes[0].set_ylabel("Number of intervals")
axes[0].set_title("ACC Arena raw sampling jitter")
axes[0].set_xticks(range(0, 8)); axes[0].legend(fontsize=8); axes[0].grid(axis="y", alpha=.25)

per_window = df.groupby("time").agg(mean_throughput=("throughput", "mean"),
                                     active_users=("traffic_type", lambda s: int((s >= MIN_TRAFFIC_TYPE).sum())))
elapsed_min = (per_window.index - per_window.index.min()) / 60
axes[1].plot(elapsed_min, per_window.mean_throughput, color="#2a9d8f", label="Mean throughput")
axes[1].set_xlabel("Elapsed time (min)")
axes[1].set_ylabel("Mean user throughput (Mbps)")
axes[1].set_title(f"Uniform {RESAMPLE_SECONDS}-s windows")
axes[1].grid(alpha=.25)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_sampling_quality.png", dpi=160); plt.show()


## Observe the complete target distribution first
No state or target value is removed here. We distinguish structural zeros from the positive-throughput tail, quantify both, and only then decide which modeling alternatives deserve comparison.

In [ ]:
import matplotlib.pyplot as plt
labels = {0: "off", 1: "idle", 2: "constant", 3: "video", 4: "gaming", 5: "http"}
full = df["throughput"]
active_mask = df.traffic_type >= MIN_TRAFFIC_TYPE
active = df.loc[active_mask, "throughput"]
positive = full[full > 0]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0,0].hist(full, bins=80, color="#457b9d")
axes[0,0].set_yscale("log")
axes[0,0].set_xlabel("Window-mean throughput (Mbps)")
axes[0,0].set_ylabel("Number of user-windows (log scale)")
axes[0,0].set_title("All states, complete throughput range")

axes[0,1].hist(np.log1p(positive), bins=80, color="#2a9d8f")
log_ticks = np.log1p([0.1, 1, 10, 100, 500])
axes[0,1].set_xticks(log_ticks, ["0.1", "1", "10", "100", "500"])
axes[0,1].set_xlabel("Positive throughput (Mbps, log1p axis)")
axes[0,1].set_ylabel("Number of user-windows")
axes[0,1].set_title("Positive values: body and tail remain visible")

by_type = (df.assign(is_zero=df.throughput == 0).groupby("traffic_type")
             .agg(samples=("throughput", "size"), zero_rate=("is_zero", "mean"),
                  mean_mbps=("throughput", "mean"), median_mbps=("throughput", "median"),
                  p95_mbps=("throughput", lambda s: s.quantile(.95)),
                  p99_mbps=("throughput", lambda s: s.quantile(.99)), max_mbps=("throughput", "max")))
by_type.index = [labels.get(int(i), str(i)) for i in by_type.index]
axes[1,0].bar(by_type.index, 100 * by_type.zero_rate, color="#e9c46a")
axes[1,0].set_xlabel("Traffic state")
axes[1,0].set_ylabel("Zero-throughput user-windows (%)")
axes[1,0].set_title("Zeros are state-dependent, not missing data")
axes[1,0].tick_params(axis="x", rotation=20)

percentiles = np.array([90, 95, 99, 99.5, 99.9])
thresholds = np.percentile(active, percentiles)
variance = (active - active.mean()) ** 2
variance_share = [100 * variance[active > t].sum() / variance.sum() for t in thresholds]
axes[1,1].plot(percentiles, variance_share, marker="o", color="#e76f51")
axes[1,1].set_xlabel("Active-throughput percentile threshold")
axes[1,1].set_ylabel("Share of total squared deviation above threshold (%)")
axes[1,1].set_title("Why MSE is sensitive to the upper tail")
axes[1,1].grid(alpha=.3)

plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_target_distribution_audit.png", dpi=160); plt.show()
print("Target summary by traffic state (Mbps):")
display(by_type.round(3))
print("Active-state percentiles (Mbps):", {p: round(float(v), 3) for p, v in zip(percentiles, thresholds)})

The upper tail strongly affects squared-error metrics, but this does **not** prove that those observations are invalid. They may be legitimate short-lived HTTP/video bursts. Therefore the primary experiment keeps them. Any active-only or train-tail restriction must be reported as a separate sensitivity analysis against the unchanged full test distribution.

## Are the upper-tail samples plausible?
We compare the top 1% of active-state throughput with the remaining active observations. Repeated users, valid traffic states, and physically coherent PRB/SINR/BLER values argue for real traffic bursts; impossible values or isolated malformed records would argue for cleaning.

In [ ]:
tail_threshold = active.quantile(.99)
active_df = df[active_mask].copy()
active_df["segment"] = np.where(active_df.throughput > tail_threshold, "top 1%", "lower 99%")
plausibility = active_df.groupby("segment").agg(
    samples=("throughput", "size"), users=("user_id", "nunique"),
    throughput_median=("throughput", "median"), throughput_max=("throughput", "max"),
    prb_median=("prb", "median"), sinr_dl_median=("sinr_dl", "median"),
    bler_median=("bler", "median"))
print(f"Active-state p99 threshold: {tail_threshold:.3f} Mbps")
display(plausibility.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
tail_rate = 100 * active_df.groupby("traffic_type")["segment"].apply(lambda s: (s == "top 1%").mean())
axes[0].bar([labels.get(int(t), str(t)) for t in tail_rate.index], tail_rate, color="#f4a261")
axes[0].set_xlabel("Active traffic type"); axes[0].set_ylabel("Samples above active-state p99 (%)")
axes[0].set_title("Which traffic states generate the tail?")

sample = active_df.sample(min(12000, len(active_df)), random_state=RANDOM_SEED)
colors = np.where(sample.throughput > tail_threshold, "#d62828", "#457b9d")
axes[1].scatter(sample.prb, sample.throughput, c=colors, s=7, alpha=.35)
axes[1].axhline(tail_threshold, color="#d62828", linestyle="--", label=f"active p99 = {tail_threshold:.2f} Mbps")
axes[1].set_xlabel("PRB utilization"); axes[1].set_ylabel("Window-mean throughput (Mbps)")
axes[1].set_yscale("symlog", linthresh=.1); axes[1].set_title("Tail observations in radio context"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_tail_plausibility.png", dpi=160); plt.show()

## Throughput by traffic type
Traffic type (0=off … 5=http) strongly conditions throughput.

In [ ]:
plot_groups = []
plot_labels = []
for traffic_type in sorted(df.traffic_type.unique()):
    values = df.loc[df.traffic_type == traffic_type, "throughput"]
    plot_groups.append(values.sample(min(5000, len(values)), random_state=RANDOM_SEED))
    plot_labels.append(labels.get(int(traffic_type), str(traffic_type)))
plt.figure(figsize=(9, 4.5))
plt.boxplot(plot_groups, labels=plot_labels, showfliers=True, flierprops={"markersize": 2, "alpha": .15})
plt.yscale("symlog", linthresh=0.1)
plt.xlabel("Traffic state")
plt.ylabel("Window-mean throughput (Mbps, symlog scale)")
plt.title("Complete throughput distribution by traffic state")
plt.grid(axis="y", alpha=.25)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_throughput_by_traffic.png", dpi=160); plt.show()

## Feature correlations
Which numeric features move with throughput?

In [ ]:
num = ["throughput", "bler", "prb", "sinr_dl", "sinr_ul", "x", "y", "z"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, frame, title in [(axes[0], df, "All traffic states"), (axes[1], df[active_mask], "Active states only (diagnostic)")]:
    corr = frame[num].corr()
    image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(num)), num, rotation=45, ha="right"); ax.set_yticks(range(len(num)), num)
    for i in range(len(num)):
        for j in range(len(num)):
            ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title)
fig.colorbar(image, ax=axes, shrink=.8, label="Pearson correlation")
fig.suptitle("Correlation changes when structural zero states are conditioned out")
plt.savefig(f"{RESULTS_DIR}/figures/01_corr_all_vs_active.png", dpi=160, bbox_inches="tight"); plt.show()

## SINR vs throughput, and user positions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))
s = df.sample(min(5000, len(df)), random_state=RANDOM_SEED)
sc = ax[0].scatter(s.sinr_dl, s.throughput, c=s.traffic_type, cmap="tab10", s=6, alpha=.5)
ax[0].set_xlabel("SINR DL (dB)"); ax[0].set_ylabel("Throughput (Mbps)"); ax[0].set_title("SINR vs throughput")
one_t = df[df.time == df.time.iloc[0]]
ax[1].scatter(one_t.x, one_t.y, c=one_t.traffic_type, cmap="tab10", s=10)
ax[1].set_xlabel("x (m)"); ax[1].set_ylabel("y (m)"); ax[1].set_title("User positions @ t0")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_sinr_positions.png", dpi=120); plt.show()

## Evidence-based modeling decisions
- `off` and `idle` are legitimate operating states with structural zeros. They remain in the primary regression task; active-only results, if run, are explicitly labeled as conditional sensitivity results.
- The high-throughput tail is rare and influential under MSE, but rarity alone is not a data-quality defect. The primary test set remains complete.
- We compare MSE-trained and robust-loss neural networks and report MAE/RMSE/R² both overall and separately for inactive, active, lower-99%, and tail slices.
- `traffic_type`, PRB, SINR and BLER carry different information; correlations are descriptive and do not determine feature removal.
- Spatial clustering motivates the Team-8 closest-user features. Neighbour throughput remains excluded to prevent target leakage.
- Final exclusions are allowed only if the EDA reveals impossible or corrupted measurements, with the rule documented and tested as an ablation.